In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

import altair as alt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from IPython.display import HTML

In [ ]:
import plotly.io as pio

pio.renderers.default = 'iframe'

In [ ]:
HTML("""
<style>
.quant-box {
background: linear-gradient(135deg,#020617,#0f172a);
padding:40px;
border-radius:15px;
color:white;
font-family:Arial;
}
.title {
font-size:42px;
font-weight:bold;
}
.sub {
color:#94a3b8;
font-size:18px;
}
</style>

<div class="quant-box">
<div class="title">Gold Research Lab</div>
<div class="sub">3D Volatility Surface • AI Regimes • Liquidity Fractals</div>
</div>
""")

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/krupalpatel07/gold-price-dynamics/GoldUSD.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

df.set_index("Date", inplace=True)

In [ ]:
df["returns"] = df["Close"].pct_change()

df["volatility"] = df["returns"].rolling(30).std()

df["momentum"] = df["Close"].pct_change(60)

df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

df["range"] = df["High"] - df["Low"]

df = df.dropna()

In [ ]:
window_sizes = [10,20,30,60]

surface = []

for w in window_sizes:
    surface.append(df["returns"].rolling(w).std())

surface = np.array(surface)

fig = go.Figure(data=[go.Surface(
    z=surface,
    x=df.index,
    y=window_sizes
)])

fig.update_layout(
title="3D Volatility Surface of Gold",
scene=dict(
    xaxis_title='Time',
    yaxis_title='Window Size',
    zaxis_title='Volatility'
),
template="plotly_dark"
)

fig.show()

In [ ]:
features = df[["returns","volatility","momentum"]]

scaler = StandardScaler()
X = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42)
df["regime"] = kmeans.fit_predict(X)

In [ ]:
fig = px.scatter(
df,
x="returns",
y="volatility",
color="regime",
title="AI Market Regime Clustering",
template="plotly_dark"
)

fig.show()

In [ ]:
fig = px.scatter(
df,
x=df.index,
y="Close",
color="regime",
title="Market Regime Timeline",
template="plotly_dark"
)

fig.show()

In [ ]:
df["vol_10"] = df["returns"].rolling(10).std()
df["vol_50"] = df["returns"].rolling(50).std()

df["fractal"] = df["vol_10"] / df["vol_50"]

In [ ]:
fig = px.line(
df,
y="fractal",
title="Liquidity Fractal Structure",
template="plotly_dark"
)

fig.show()

In [ ]:
df["year"] = df.index.year

fig = px.line(
df,
x=df.index,
y=["Close","volatility"],
animation_frame="year",
title="Gold Market Evolution",
template="plotly_dark"
)

fig.show()

In [ ]:
df["alpha"] = df["momentum"] - df["volatility"]

In [ ]:
fig = px.scatter(
df,
x="alpha",
y="returns",
color="regime",
title="Alpha Signal vs Returns",
template="plotly_dark"
)

fig.show()

In [ ]:
df1 = df[-4999:]

In [ ]:
chart = alt.Chart(df1.reset_index()).mark_line().encode(
    x="Date:T",
    y="Close:Q",
    color="regime:N"
).properties(
    width=900,
    height=400,
    title="Gold Price with AI Regimes"
)

chart.interactive()

In [ ]:
HTML(f"""
<div style='display:flex'>

<div style='background:#111827;padding:20px;margin:10px;border-radius:10px;color:white'>
<h3>Avg Volatility</h3>
<h1>{round(df.volatility.mean(),4)}</h1>
</div>

<div style='background:#111827;padding:20px;margin:10px;border-radius:10px;color:white'>
<h3>Total Regimes</h3>
<h1>{df.regime.nunique()}</h1>
</div>

<div style='background:#111827;padding:20px;margin:10px;border-radius:10px;color:white'>
<h3>Max Return</h3>
<h1>{round(df.returns.max(),3)}</h1>
</div>

</div>
""")